In [14]:
import glob
for p in glob.glob('/kaggle/input/**/*.csv', recursive=True):
    print(p)

/kaggle/input/datasets/shravaniuk/phase-comparison-data/test_predictions_longformer.csv
/kaggle/input/datasets/shravaniuk/phase-comparison-data/test_predictions.csv


In [15]:
import pandas as pd

bert = pd.read_csv('/kaggle/input/datasets/shravaniuk/phase-comparison-data/test_predictions.csv',
                   dtype={'essay_id': str})
lf   = pd.read_csv('/kaggle/input/datasets/shravaniuk/phase-comparison-data/test_predictions_longformer.csv',
                   dtype={'essay_id': str})

merged = lf.merge(bert, on='essay_id', suffixes=('_lf','_bert'))
print("Identical test set   :", set(lf.essay_id) == set(bert.essay_id))
print("Overlap essays       :", len(merged))
print("True-score agreement :", (merged.true_score_lf == merged.true_score_bert).mean())

Identical test set   : False
Overlap essays       : 1489
True-score agreement : 1.0


In [16]:
import pandas as pd

bert = pd.read_csv('/kaggle/input/datasets/shravaniuk/phase-comparison-data/test_predictions.csv',            dtype=str)
lf   = pd.read_csv('/kaggle/input/datasets/shravaniuk/phase-comparison-data/test_predictions_longformer.csv', dtype=str)

print("Same number of rows :", len(bert) == len(lf) == 1559)

# Check they're in the same order by comparing true scores row-by-row
same_order = (bert['true_score'].astype(int).values ==
              lf['true_score'].astype(int).values).mean()
print("Row-by-row true-score agreement :", same_order)

Same number of rows : True
Row-by-row true-score agreement : 1.0


In [17]:
comp = pd.DataFrame({
    'true'     : bert['true_score'].astype(int).values,
    'bert_pred': bert['pred_score'].astype(int).values,
    'lf_pred'  : lf['pred_score'].astype(int).values,
})
comp['bert_error'] = (comp['true'] - comp['bert_pred']).abs()
comp['lf_error']   = (comp['true'] - comp['lf_pred']).abs()
print(comp.head())
print("BERT mean error :", comp['bert_error'].mean())
print("LF mean error   :", comp['lf_error'].mean())

   true  bert_pred  lf_pred  bert_error  lf_error
0     2          3        3           1         1
1     4          4        4           0         0
2     5          4        4           1         1
3     4          4        4           0         0
4     4          4        4           0         0
BERT mean error : 0.3976908274534958
LF mean error   : 0.3354714560615779


In [18]:
from sklearn.metrics import cohen_kappa_score

def metrics(true, pred):
    return {
        'QWK'     : round(cohen_kappa_score(true, pred, weights='quadratic'), 4),
        'MAE'     : round((abs(true - pred)).mean(), 4),
        'Exact'   : round((true == pred).mean(), 4),
        'Adjacent': round((abs(true - pred) <= 1).mean(), 4),
    }

summary = pd.DataFrame({
    'BERT'      : metrics(comp['true'], comp['bert_pred']),
    'Longformer': metrics(comp['true'], comp['lf_pred']),
}).T
print(summary)

# How often does each model beat the other, essay by essay?
lf_better   = (comp['lf_error']   < comp['bert_error']).sum()
bert_better = (comp['bert_error'] < comp['lf_error']).sum()
tie         = (comp['lf_error']   == comp['bert_error']).sum()
print(f"\nLongformer closer : {lf_better}")
print(f"BERT closer       : {bert_better}")
print(f"Tie               : {tie}")

               QWK     MAE   Exact  Adjacent
BERT        0.8283  0.3977  0.6216    0.9814
Longformer  0.8555  0.3355  0.6754    0.9897

Longformer closer : 227
BERT closer       : 132
Tie               : 1200
